***Total: 42 points***

Complete this homework by writing R code to complete the following tasks. Keep in mind:

i. Empty chunks have been included where code is required
ii. This homework requires use of data files:

  - `BRCA.genome_wide_snp_6_broad_Level_3_scna.seg` (Problems 1, 2)
  - `GIAB_highconf_v.3.3.2.vcf.gz` (Problem 3)

   Download these from https://drive.google.com/drive/folders/1zNxwqrwIBKdrlIqED7caw72nYy9tW0-c?usp=sharing
  
iv. You will be graded on your code and output results. The assignment is worth 42 points total; partial credit can be awarded.

For additional resources, please refer to these links:  
Problems 1 & 2:  
  - https://www.bioconductor.org/packages/devel/bioc/vignettes/plyranges/inst/doc/an-introduction.html
  - https://bioconductor.org/packages/release/bioc/vignettes/GenomicRanges/inst/doc/GenomicRangesIntroduction.html  
Problem 3:  
  - https://bioconductor.org/packages/release/bioc/vignettes/Rsamtools/inst/doc/Rsamtools-Overview.pdf  
Problem 4: 
  - https://bioconductor.org/packages/release/bioc/vignettes/VariantAnnotation/inst/doc/VariantAnnotation.pdf  

# Problem 1: Overlaps between genomic regions and copy number alterations. (14 points total)

### Preparation
Load copy number segment results as shown in *2.1 BED format* of *Lecture16_GenomicData.Rmd*. You will use the same file as in the lecture notes, `BRCA.genome_wide_snp_6_broad_Level_3_scna.seg`. Here is code to get you started.

In [2]:
#load packages
suppressPackageStartupMessages({
    library(tidyverse)
    library(GenomicRanges)
    library(plyranges)
    library(VariantAnnotation)
})

In [7]:
segs <- read_tsv("BRCA.genome_wide_snp_6_broad_Level_3_scna.seg") %>%
  janitor::clean_names() %>%
  print()

Rows: 284458 Columns: 6
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (1): Sample
dbl (5): Chromosome, Start, End, Num_Probes, Segment_Mean

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


# A tibble: 284,458 × 6
   sample                       chromosome  start    end num_probes segment_mean
   <chr>                             <dbl>  <dbl>  <dbl>      <dbl>        <dbl>
 1 TCGA-3C-AAAU-10A-01D-A41E-01          1 3.22e6 9.57e7      53225       0.0055
 2 TCGA-3C-AAAU-10A-01D-A41E-01          1 9.57e7 9.57e7          2      -1.66  
 3 TCGA-3C-AAAU-10A-01D-A41E-01          1 9.57e7 1.67e8      24886       0.0053
 4 TCGA-3C-AAAU-10A-01D-A41E-01          1 1.67e8 1.67e8          3      -1.10  
 5 TCGA-3C-AAAU-10A-01D-A41E-01          1 1.67e8 1.82e8       9213      -0.0008
 6 TCGA-3C-AAAU-10A-01D-A41E-01          1 1.82e8 1.82e8          6      -1.20  
 7 TCGA-3C-AAAU-10A-01D-A41E-01          1 1.82e8 2.01e8      12002       0.0055
 8 TCGA-3C-AAAU-10A-01D-A41E-01          1 2.01e8 2.01e8          2      -1.42  
 9 TCGA-3C-AAAU-10A-01D-A41E-01          1 2.01e8 2.48e8      29781      -0.0004
10 TCGA-3C-AAAU-10A-01D-A41E-01          2 4.84e5 5.15e7      30300       0.0044
# … 

In [36]:
myGRange <- data.frame(seqnames = "8", start = 128746347, end = 128755810) %>% as_granges()

In [37]:
segs <- read.delim("BRCA.genome_wide_snp_6_broad_Level_3_scna.seg", as.is = TRUE)
mode(segs$Chromosome) <- "character" 
segs[segs$Chromosome == 23, "Chromosome"] <- "X"

segs.gr <- segs %>%
  GRanges()


### a. Find the segments in `segs.gr` that have *any* overlap with the region `chr8:128,746,347-128,755,810` (4 points)
Print out the first five unique TCGA IDs.

In [50]:
seqinfo <- Seqinfo(genome = "hg19")
seqinfo <- keepStandardChromosomes(seqinfo) 
seqlevelsStyle(seqinfo) <- "NCBI"
invisible(seqinfo)

Warning message in (function (seqlevels, genome, new_style) :
“cannot switch some of hg19's seqlevels from UCSC to NCBI style”


In [51]:
slen <- seqlengths(seqinfo) # get the length of the chromosomes
tileWidth <- 500000 # tile size of 500kb
tiles <- tileGenome(seqlengths = slen, tilewidth = tileWidth,
                    cut.last.tile.in.chrom = TRUE)
invisible(tiles)

In [52]:
tiles.subset <- tiles[seqnames(tiles) == "17" & start(tiles) >= 35500000 & end(tiles) <= 37000000]
invisible(tiles.subset)

In [53]:
segs.overlap <- find_overlaps(segs.gr, tiles.subset)  # arguments: find_overlaps(query, subject)
segs.overlap[1:5]

GRanges object with 5 ranges and 3 metadata columns:
      seqnames            ranges strand |                 Sample Num_Probes
         <Rle>         <IRanges>  <Rle> |            <character>  <integer>
  [1]       17   987221-73296953      * | TCGA-3C-AAAU-10A-01D..      33859
  [2]       17   987221-73296953      * | TCGA-3C-AAAU-10A-01D..      33859
  [3]       17   987221-73296953      * | TCGA-3C-AAAU-10A-01D..      33859
  [4]       17 25270517-73296953      * | TCGA-3C-AAAU-01A-11D..      24226
  [5]       17 25270517-73296953      * | TCGA-3C-AAAU-01A-11D..      24226
      Segment_Mean
         <numeric>
  [1]       0.0088
  [2]       0.0088
  [3]       0.0088
  [4]       0.1856
  [5]       0.1856
  -------
  seqinfo: 23 sequences from an unspecified genome; no seqlengths

### b. Find the mean of the `Segment_Mean` values for copy number segments that have *any* overlap with the region chr17:37,842,337-37,886,915. (4 points)

### c. Find the patient sample distribution of copy number for `PIK3CA` (hg19). (6 points)
Find the counts of samples with deletion (D; `Segment_Mean < -0.3`), neutral (N; `Segment_Mean >= -0.3 & Segment_Mean <= 0.3`), gain (G; `Segment_Mean > 0.3`) segments that have `any` overlap with `PIK3CA` gene coordinates.  


# Problem 2: Frequency of copy number alteration events within genomic regions. (12 points total) 

This problem will continue to use the copy number data stored in `segs.gr`.

### a. Create a genome-wide tile of 1Mb windows for the human genome (`hg19`). (6 points)
See *3.1 Tiling the genome* of *Lecture16_GenomicData.Rmd* for hints.


### b. Find the 1Mb window with the most frequent overlapping deletions. (6 points)
Find the 1Mb windows with `any` overlap with deletion copy number segments. Assume a deletion segment is defined as a segment in `segs.gr` having `Segment_Mean < -0.3`. 

Return one of the 1Mb window `Granges` entry with the highest frequency (count) of deletion segments.

Hint: Subset the `segs.gr` to only rows with `Segment_Mean < -0.3`. 

# Problem 3: Reading and annotating genomic variants (16 points total)

### Preparation

In [6]:
vcfFile <- "GIAB_highconf_v.3.3.2.vcf.gz"

### a. Load variant data from VCF file `GIAB_highconf_v.3.3.2.vcf.gz` for `chr8:128,700,000-129,000,000`. (4 points)
Note: use genome build `hg19`.

### b. Combine the fields of the VCF genotype information into a table. (4 points)
You may use your choice of data objects (e.g. `data.frame`).

### c. Retrieve the following information at chr8:128747953. (8 points)
Print out the SNP ID (i.e. "rs ID"), reference base (`REF`), alterate base (`ALT`), genotype (`GT`), depth (`DP`), allele depth (`ADALL`), phase set (`PS`).

Hints: 

  i. `REF` and `ALT` are in the output of `rowRanges(vcf)`. See Section `3a` in `Lecture16_VariantCalls.ipynb` 
  ii. To get the sequence of `DNAString`, use `as.character(x)`.  
  ii. To get the sequence of `DNAStringSet`, use `as.character(unlist(x))`. 
  iii. To expand a list of information for `geno`, use `unlist(x)`.  

  